# 09 — Growth, limits and running several jobs at once

This is the notebook that decides whether the rest is trustworthy, because it is where we go
looking for the places a single machine **loses**.

Three questions:

1. **What happens as the data grows?** Same query, increasing sizes.
2. **Where does one machine actually give up?** Not extrapolated — measured.
3. **What about many jobs at once?** One machine does one thing at a time. A cluster does not.

### On the crossover point

We are not going to claim a crossover point we did not measure. On a machine this size DuckDB
will very likely still be ahead at the largest size we can fit, and saying "but they cross
eventually, trust me" is exactly the sort of hand-wave that deserves to be picked apart.

So instead of guessing, we measure something real: **the memory at which the job stops working
at all**. That gives a boundary with a mechanism behind it, and a scaling rule anyone can check:

> the job needed roughly *N* of memory at this size — so a machine with 8× the memory handles
> roughly 8× the data, and past that you need more than one machine.

Same conclusion, but it is arithmetic from a measurement rather than a vibe.

In [ ]:
import sys, os, time, threading
sys.path.insert(0, os.path.abspath(".."))
from bench import config as C, datagen, engines, report
from bench.harness import Case, Bench
import duckdb, pandas as pd
import matplotlib.pyplot as plt

paths = datagen.paths(C.MAIN_SIZE)
duck  = engines.get_duckdb()
spark = engines.get_spark()
engines.attach(duck, spark, paths)
bench = Bench(duck, spark, "09_limits", C.MAIN_SIZE)
print(f"Ready. Sizes available: {[C.human(s) for s in C.SIZES]}")

## 1. What happens as the data grows

One representative query — a filtered join with a group-by, the shape of most reporting work —
run at every size we generated.

Watch the **shape** of the lines, not just the gap. If the gap stays wide at the right-hand
edge, we have not yet reached the size where distributing the work starts to pay for itself.

In [ ]:
SWEEP_SQL = """
SELECT c.country, count(*) AS orders, round(sum(s.amount),2) AS revenue
FROM sales s JOIN customers c ON c.customer_id = s.customer_id
WHERE s.channel = 'web' AND s.amount IS NOT NULL
GROUP BY 1 ORDER BY revenue DESC
"""

sweep = []
for n in C.SIZES:
    p = datagen.paths(n)
    engines.attach(duck, spark, p, extras=False)
    b2 = Bench(duck, spark, "_sweep", n)
    r, _, _ = b2.run(Case(f"sweep_{n}", f"Sweep at {C.human(n)}", "Sweep",
                          sql=SWEEP_SQL.strip()), quiet=True)
    sweep.append({"rows": n, "DuckDB": r["duckdb_s"], "Spark": r["spark_s"]})
    msg = f"   {C.human(n):>13} rows   DuckDB {r['duckdb_s']:7.3f}s"
    if r["spark_s"]:
        msg += f"   Spark {r['spark_s']:7.2f}s   ({r['ratio']}x)"
    print(msg)

engines.attach(duck, spark, paths)          # put things back
sw = pd.DataFrame(sweep)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(sw["rows"], sw["DuckDB"], "o-", lw=2.5, ms=8, color=report.DUCK_C, label="DuckDB")
if sw["Spark"].notna().any():
    ax.plot(sw["rows"], sw["Spark"], "s-", lw=2.5, ms=8, color=report.SPARK_C, label="Spark")

# Where would this job land if it grew?
biggest = C.SIZES[-1]
for mult, label in [(10, "10x growth"), (100, "100x growth")]:
    ax.axvline(biggest * mult, color="#999", ls=":", lw=1.4)
    ax.text(biggest * mult, ax.get_ylim()[1] * 0.94, f"  {label}", fontsize=9,
            color="#555", rotation=90, va="top")

ax.set_xscale("log"); ax.minorticks_off()
ticks = list(C.SIZES) + [biggest * 10, biggest * 100]
ax.set_xticks(ticks); ax.set_xticklabels([C.human(t) for t in ticks], fontsize=8)
ax.set_xlabel("Number of sales rows"); ax.set_ylabel("Seconds to answer")
ax.set_ylim(bottom=0)
ax.set_title("The same question, as the data grows", fontsize=12.5, weight="bold", pad=12)
ax.legend(); ax.grid(alpha=.25)
for sp in ("top", "right"): ax.spines[sp].set_visible(False)
plt.tight_layout(); plt.show()

print("The dotted lines answer the objection this whole project exists for:")
print('"but it might get big later". Grant the growth, then look at where it lands.')

## 2. Where does one machine give up?

We give DuckDB a smaller and smaller memory budget and re-run a genuinely heavy job — joins
plus window functions over the whole table.

Two things can happen, and both are worth showing:

* **It slows down.** It starts using the disk as overflow space, like clearing a full desk by
  stacking papers on the floor. Slower, but it still finishes.
* **It stops.** Below some point there is not enough room to work at all.

The second one is the number we want. It is the honest right-hand edge of the argument.

In [ ]:
HEAVY = """
SELECT segment, count(*) AS n, round(avg(spend_so_far),2) AS avg_spend
FROM (
  SELECT c.segment,
         sum(s.amount) OVER (PARTITION BY s.customer_id ORDER BY s.sale_ts) AS spend_so_far
  FROM sales s JOIN customers c ON c.customer_id = s.customer_id
  WHERE s.amount IS NOT NULL
) t GROUP BY 1 ORDER BY 1
"""

limits = [C.ENGINE_MEMORY_MB, 1024, 512, 256, 128, 64, 32, 16]
limits = sorted({int(x) for x in limits}, reverse=True)
wall, baseline = [], None

print(f"Re-running a heavy job on a smaller and smaller desk ({C.human(C.MAIN_SIZE)} rows)\n")
for mb in limits:
    try:
        tight = duckdb.connect(config={"memory_limit": f"{mb}MB", "threads": str(C.ENGINE_THREADS)})
        tight.execute(f"CREATE OR REPLACE VIEW sales AS SELECT * FROM read_parquet('{paths['sales']}')")
        tight.execute(f"CREATE OR REPLACE VIEW customers AS SELECT * FROM read_parquet('{paths['customers']}')")
        t0 = time.perf_counter(); tight.execute(HEAVY).df(); took = time.perf_counter() - t0
        tight.close()
        baseline = baseline or took
        wall.append({"memory": f"{mb} MB", "seconds": round(took, 2),
                     "vs roomy": f"{took/baseline:.1f}x", "outcome": "finished"})
        print(f"   {mb:>6} MB  ->  {took:7.2f} s")
    except Exception as e:
        wall.append({"memory": f"{mb} MB", "seconds": None, "vs roomy": "-",
                     "outcome": "STOPPED: " + str(e)[:60]})
        print(f"   {mb:>6} MB  ->  STOPPED")

wdf = pd.DataFrame(wall); display(wdf)

In [ ]:
stopped = wdf[wdf["outcome"].str.startswith("STOPPED")]
if len(stopped):
    floor_mb = int(stopped.iloc[0]["memory"].split()[0])
    survived = wdf[wdf["outcome"] == "finished"]
    lowest_ok = int(survived.iloc[-1]["memory"].split()[0]) if len(survived) else None
    print(f"THE WALL\n")
    print(f"   This job finished with {lowest_ok} MB and stopped at {floor_mb} MB,")
    print(f"   on {C.human(C.MAIN_SIZE)} rows.\n")
    if lowest_ok:
        per_m = lowest_ok / (C.MAIN_SIZE / 1_000_000)
        print(f"   That is roughly {per_m:.1f} MB of working memory per million rows.\n")
        print("   Scaling that up -- state this out loud as arithmetic, not measurement:")
        for gb in (16, 64, 128):
            print(f"      a {gb:>3} GB machine handles roughly "
                  f"{gb*1024/per_m/1000:,.0f} million rows of this job")
        print("\n   Above that you need more than one machine. That is what Spark is for,")
        print("   and proving exactly where the line falls needs real data on real hardware.")
else:
    print("It never stopped at these sizes -- which is itself the finding: this job is")
    print("nowhere near needing a cluster. Re-run with BENCH_QUICK=0 to push it harder.")

## 3. Several jobs at once

The most legitimate reason to run a shared cluster.

One machine does one job at a time. A cluster serves many people at once. Here we fire four
different queries simultaneously and compare the total wall-clock time against running them
one after another.

If DuckDB's advantage shrinks here, that is a real finding and it belongs in the argument.
It is also the second axis of the rule of thumb: **not just how big, but how many at once.**

In [ ]:
JOBS = [
 "SELECT channel, count(*) AS n, round(sum(amount),2) AS rev FROM sales GROUP BY 1 ORDER BY 1",
 "SELECT c.country, count(*) AS n FROM sales s JOIN customers c ON c.customer_id=s.customer_id GROUP BY 1 ORDER BY 1",
 "SELECT region, round(avg(amount),2) AS a FROM sales WHERE amount IS NOT NULL GROUP BY 1 ORDER BY 1",
 "SELECT count(DISTINCT customer_id) AS c, count(DISTINCT product_id) AS p FROM sales",
]

def duck_seq():
    for q in JOBS: duck.execute(q).df()

def duck_par():
    ths = []
    for q in JOBS:
        def run(qq=q):
            cur = duck.cursor()          # each thread gets its own cursor
            cur.execute(qq).df()
        t = threading.Thread(target=run); t.start(); ths.append(t)
    for t in ths: t.join()

def spark_seq():
    for q in JOBS: spark.sql(q).toPandas()

def spark_par():
    ths = []
    for q in JOBS:
        def run(qq=q): spark.sql(qq).toPandas()
        t = threading.Thread(target=run); t.start(); ths.append(t)
    for t in ths: t.join()

def timeit(fn):
    fn()                                  # warm-up
    t0 = time.perf_counter(); fn(); return time.perf_counter() - t0

rows = [{"engine": "DuckDB", "one after another (s)": round(timeit(duck_seq), 3),
         "all four at once (s)": round(timeit(duck_par), 3)}]
if spark:
    rows.append({"engine": "Spark", "one after another (s)": round(timeit(spark_seq), 3),
                 "all four at once (s)": round(timeit(spark_par), 3)})

cdf = pd.DataFrame(rows)
cdf["speed-up from running together"] = (cdf["one after another (s)"] /
                                          cdf["all four at once (s)"]).round(2)
display(cdf)
print("A number near 1.0 means running jobs together bought nothing.")
print("Higher means the engine overlapped the work.")
print()
print("Read this one carefully. It is measured on ONE machine, so it shows how well each")
print("engine overlaps concurrent work locally -- NOT what a real multi-machine cluster")
print("does with many users. That is a genuine advantage of a cluster that this cannot show.")

## What this notebook is for

Everything else in the project shows DuckDB doing well. This one exists to mark the edges:

* the sweep shows whether the gap is closing at the sizes we can reach
* the wall shows where a single machine genuinely stops, with a scaling rule from it
* the concurrency test shows the case a shared cluster is actually built for

A comparison that only ever points one way should not be believed. These are the parts that
make the rest credible.

In [ ]:
bench.rows.extend([
    {"notebook": "09_limits", "id": "X1", "operation": "Memory wall (see table above)",
     "category": "Limits", "rows": C.MAIN_SIZE, "identical_sql": False,
     "why": "Where one machine stops working", "note": "see notebook 09",
     "duckdb_s": None, "spark_s": None, "ratio": None, "same_answer": None,
     "duckdb_error": None, "spark_error": None},
])
bench.save()
engines.stop_spark()